# 实验 7 · 案例 1：房源信息爬取

使用 Scrapy 的 **Spider → Item → Pipeline** 流程，采集楼盘名称、地址、均价和总价，保存为 CSV

## 1. 环境与项目创建

In [ ]:
%pip install scrapy pandas itemadapter

In [10]:
import importlib.util
import subprocess
import sys

missing = [p for p in ("scrapy", "pandas") if importlib.util.find_spec(p) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)

In [11]:
from pathlib import Path

project = Path.cwd() / "house"
if not (project / "scrapy.cfg").exists():
    subprocess.run(
        [sys.executable, "-m", "scrapy", "startproject", "house", str(project)],
        check=True,
    )
print(project)

c:\Users\ZhuanZ（无密码）\Desktop\BigDataCourse\大数据采集与处理-学生实验环境\workspace\bigdata-collection-labs-20261017\大数据采集与处理实验报告-7-邢贺-112412060139\house


## 2. 定义 Item

In [12]:
%%writefile house/house/items.py
import scrapy


class ShellsItem(scrapy.Item):
    house = scrapy.Field()
    address = scrapy.Field()
    price = scrapy.Field()
    total = scrapy.Field()

Overwriting house/house/items.py


## 3. 编写 Spider：分页与 XPath 解析

In [13]:
%%writefile house/house/spiders/shell.py
import re
import scrapy
from house.items import ShellsItem


class ShellsSpider(scrapy.Spider):
    name = "shell"
    allowed_domains = ["www.bspider.top", "bspider.top"]
    start_urls = [f"http://www.bspider.top/fangke/pg{i}" for i in range(1, 4)]

    def parse(self, response):
        # 每个 li 对应一条房源记录
        rows = response.xpath('//ul[@class="resblock-list-wrapper"]/li')
        for row in rows:
            item = ShellsItem()
            item["house"] = row.xpath('div/div[1]/a/text()').get(default="").strip()
            item["address"] = row.xpath('div/a[1]/@title').get(default="").strip()

            # string() 合并节点中的文本，去掉空白字符
            price = row.xpath('string(.//div/div[4]/div[1])').get(default="")
            item["price"] = re.sub(r"\s+", "", price)

            total = row.xpath('div/div[4]/div[2]/text()').get(default="")
            item["total"] = total.replace("总价:", "").replace("总价：", "").strip()

            if item["house"]:
                yield item

Overwriting house/house/spiders/shell.py


## 4. Pipeline：保存 CSV

In [14]:
%%writefile house/house/pipelines.py
import csv
from pathlib import Path
from itemadapter import ItemAdapter


class CsvPipeline:
    def open_spider(self, spider):
        self.file = (Path.cwd() / "shells.csv").open(
            "w", encoding="utf-8-sig", newline=""
        )
        self.writer = csv.DictWriter(
            self.file, fieldnames=["house", "address", "price", "total"]
        )
        self.writer.writeheader()

    def process_item(self, item, spider):
        self.writer.writerow(ItemAdapter(item).asdict())
        return item

    def close_spider(self, spider):
        self.file.close()

Overwriting house/house/pipelines.py


## 5. 配置与启动

In [15]:
%%writefile house/house/settings.py
BOT_NAME = "house"
SPIDER_MODULES = ["house.spiders"]
NEWSPIDER_MODULE = "house.spiders"

ROBOTSTXT_OBEY = True
DOWNLOAD_DELAY = 0.5
DOWNLOAD_TIMEOUT = 20
LOG_LEVEL = "INFO"

ITEM_PIPELINES = {"house.pipelines.CsvPipeline": 300}

Overwriting house/house/settings.py


In [16]:
result = subprocess.run(
    [sys.executable, "-m", "scrapy", "crawl", "shell"],
    cwd=project,
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
)
print(result.stderr[-3500:])
result.check_returncode()

pidermiddlewares.httperror.HttpErrorMiddleware',
 'scrapy.spidermiddlewares.referer.RefererMiddleware',
 'scrapy.spidermiddlewares.urllength.UrlLengthMiddleware',
 'scrapy.spidermiddlewares.depth.DepthMiddleware',
 'scrapy.spidermiddlewares.metacopy.MetaCopyDetectionMiddleware']
2026-11-09 11:08:20 [scrapy.middleware] INFO: Enabled item pipelines:
['house.pipelines.CsvPipeline']
2026-11-09 11:08:20 [py.warnings] WARNING: e:\miniconda\envs\bigdata\Lib\site-packages\scrapy\pipelines\__init__.py:60: ScrapyDeprecationWarning: CsvPipeline.open_spider() requires a spider argument, this is deprecated and the argument will not be passed in future Scrapy versions. If you need to access the spider instance you can save the crawler instance passed to from_crawler() and use its spider attribute.
  self._check_mw_method_spider_arg(mw.open_spider)

2026-11-09 11:08:20 [py.warnings] WARNING: e:\miniconda\envs\bigdata\Lib\site-packages\scrapy\pipelines\__init__.py:63: ScrapyDeprecationWarning: CsvPipe

## 6. 查看采集结果

In [17]:
import pandas as pd

csv_path = project / "shells.csv"
df = pd.read_csv(csv_path)
display(df.head(10))
print("记录数：", len(df))

,house,address,price,total
0,华南城巴南华府,龙洲湾隧道入口旁,10000元/㎡(均价),NaN
1,蓝光公园华府,富洲路（南方苹果派旁）,20000元/㎡(均价),总价168-223(万/套)
2,金地自在城,金地自在城一期营销中心华岩酒店用品城旁,13000元/㎡(均价),总价95-172(万/套)
3,中海春华九里,彩云湖CLD中央居住区的核心腹地，火炬大道中段,21000元/㎡(均价),总价230-270(万/套)
4,龙兴国际生态新城,渝北区普福大道第一沟旁,13500元/㎡(均价),总价230-350(万/套)
5,碧桂园中俊天玺,渝北区湖彩路附近,价格待定,NaN
6,未来城市,龙兴镇两江大道龙脑山公交站旁,价格待定,NaN
7,金科南山,龙兴镇两江大道龙脑山公交站旁,20000元/㎡(均价),总价100(万/套)
8,龙兴国际生态新城,渝北区普福大道第一沟旁,13500元/㎡(均价),总价125(万/套)
9,洺悦城,新城红河南路888号,14800元/㎡(均价),总价40(万/套)


记录数： 30
